In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

df = pd.read_csv(f"{DATA_PATH}/all_data.csv", sep="\t")
df.columns

In [ ]:
cols_to_use = ["Sex", "Age", "Tumor", "CC", "PreOP CTx", "Thermoablation", "sPCI", "pPCI"]
df = df[cols_to_use].copy()

In [ ]:
df.head()

## Variable overview

**Binary predictors**
- `Sex` (0/1, recoded from original 1/2)
- `CC` (0/1; only CC0 and CC1 retained)
- `Thermoablation` (0/1)
- `PreOP_CTx` (0/1; binarised: ≥1 → 1)

**Continuous predictor**
- `Age`

**Nominal predictor — one-hot encoded**
- `Tumor` (types 2–7; reference category = type_1, n=122, most frequent; type_8 excluded)
  - `Tumor_type_2`, `Tumor_type_3`, `Tumor_type_4`, `Tumor_type_5`, `Tumor_type_6`, `Tumor_type_7`

**Outcome variables**
- `raw_diff` = sPCI − pPCI  (signed difference)
- `abs_diff` = |raw_diff|   (absolute difference)


In [ ]:
# data wrangling
df['Sex'] = df['Sex']-1
df['PreOP CTx'] = df['PreOP CTx'].apply(lambda x: 1 if x >= 1 else x)

keep_tumor = [1, 5, 4, 6, 7, 3, 2]  # remove 8 if you decide to drop it

# Filter rows
df = df[df["Tumor"].isin(keep_tumor)].copy()
df['Tumor'] = df['Tumor'].apply(lambda x: f"type_{x}")

df = df[(df['CC']==0) | (df['CC']==1)].reset_index(drop=True)  # keep only CC0 and CC1


In [ ]:
df.shape

In [ ]:
# --- 1. Rename columns ---
df = df.rename(columns={
    "PreOP CTx" : "PreOP_CTx",
})

In [ ]:
# --- 5. One-hot encode Tumor (most frequent as reference) ---
# Tumor "1" (n=122) is the natural reference category
df = pd.get_dummies(df, columns=["Tumor"], drop_first=False, dtype=int)
df = df.drop(columns=["Tumor_type_1"], inplace=False)  # explicitly set Tumor_1 as reference

# --- 6. Create outcome variables ---
df["raw_diff"] = df["sPCI"] - df["pPCI"]
df["abs_diff"] = np.abs(df["raw_diff"])


In [ ]:
feature_cols = [
    'Sex', 'Age', 'CC', 'PreOP_CTx', 'Thermoablation', 'Tumor_type_2', 'Tumor_type_3', 'Tumor_type_4', 'Tumor_type_5',
       'Tumor_type_6', 'Tumor_type_7'
]

X     = df[feature_cols]
y_raw = df["raw_diff"]
y_abs = df["abs_diff"]

print(f"Final feature matrix: {X.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, outcome, label in zip(axes, [y_raw, y_abs], ["raw_diff", "abs_diff"]):
    ax.hist(outcome, bins=25, color="steelblue", edgecolor="white")
    ax.set_title(f"Distribution of {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Frequency")
    stat, p = stats.shapiro(outcome)
    ax.set_title(f"{label}\nShapiro-Wilk: W={stat:.3f}, p={p:.3f}")

plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(X)  # if not already included
model = sm.OLS(y_raw, X).fit()

print(model.summary())

In [ ]:
fitted = model.fittedvalues
residuals = model.resid

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(fitted, residuals, alpha=0.6)
plt.axhline(0, linestyle='--')
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted")
plt.show()

In [ ]:
plt.hist(residuals, bins=25, edgecolor="white")
plt.title("Residuals distribution")
plt.show()

In [ ]:
import statsmodels.api as sm

sm.qqplot(residuals, line='45')
plt.title("Q-Q plot")
plt.show()

In [ ]:
from scipy import stats
stat, p = stats.shapiro(residuals)
print(f"W={stat:.3f}, p={p:.3f}")

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(residuals, X)

print(f"LM p-value: {lm_pvalue:.4f}")